# 07 · Playwright: páginas protegidas con login

A veces no hay API y la información vive en páginas que **exigen login**. En
`tienda-virtual`, `proxy.ts` protege `/clientes`, `/ordenes`, `/carrito` y `/admin/*`:
sin cookie de sesión válida, cualquier petición redirige (**HTTP 307**) a `/login`.

Con [Playwright](https://playwright.dev/python/) automatizamos un navegador real:
abrimos `/login`, completamos el formulario (`admin@tienda.local` / `admin123`) y a
partir de ahí el navegador conserva la cookie de sesión. Guardamos el `storage_state`
para reutilizar la sesión sin volver a loguear.

Requisitos: `tienda-virtual` corriendo y `playwright install chromium` ya ejecutado.

In [1]:
import csv
import os

import requests
from playwright.async_api import async_playwright

BASE = "http://localhost:3000"
USUARIO = "admin@tienda.local"
CLAVE = "admin123"
UA = "MineriaWeb-2026-2/1.0 (scraping con login)"

# Sin sesión, /ordenes redirige a /login (no seguimos el redirect para verlo):
r = requests.get(f"{BASE}/ordenes", allow_redirects=False, timeout=15)
print("GET /ordenes sin sesión ->", r.status_code, "->", r.headers.get("location"))

GET /ordenes sin sesión -> 307 -> /login?next=%2Fordenes


## Iniciar sesión con Playwright

Rellenamos el formulario por el atributo `name` de cada input y esperamos a que el
`submit` nos deje en la home ya autenticados.

In [2]:
playwright = await async_playwright().start()
navegador = await playwright.chromium.launch(headless=True)
contexto = await navegador.new_context(user_agent=UA)
pagina = await contexto.new_page()

await pagina.goto(f"{BASE}/login", wait_until="networkidle")
await pagina.fill("input[name='email']", USUARIO)
await pagina.fill("input[name='password']", CLAVE)
await pagina.click("button[type='submit']")
await pagina.wait_for_url(f"{BASE}/", wait_until="networkidle")

print("URL tras el login:", pagina.url)
cookies = await contexto.cookies()
print("Cookies presentes:", [c["name"] for c in cookies])

URL tras el login: http://localhost:3000/
Cookies presentes: ['tienda_session']


## Guardar la sesión (`storage_state`) para reutilizarla

In [3]:
DATA_DIR = os.path.join(os.getcwd(), "..", "data")
os.makedirs(DATA_DIR, exist_ok=True)
STATE_PATH = os.path.join(DATA_DIR, "playwright_storage_state.json")

await contexto.storage_state(path=STATE_PATH)
print("Sesión guardada en", STATE_PATH)

Sesión guardada en /Users/erichuiza/Documents/pucp/miería web/2026-2/dev/sesion-de-clase-03/notebooks/../data/playwright_storage_state.json


## Recorrer el listado paginado protegido `/ordenes`

Ya con sesión, `/ordenes` responde con HTML. Cada `<article data-orden-id>` puede
traer reseñas de entrega embebidas como `<li data-tipo="post_compra">` con sus datos
en atributos `data-*`. Paramos cuando el enlace **Siguiente** deja de ser un `<a>`.

In [4]:
resenas = []
numero_pagina = 1
while True:
    await pagina.goto(f"{BASE}/ordenes?page={numero_pagina}", wait_until="networkidle")

    articulos = pagina.locator("article[data-orden-id]")
    n = await articulos.count()
    if n == 0:
        break

    for i in range(n):
        art = articulos.nth(i)
        orden_id = await art.get_attribute("data-orden-id")
        li = art.locator("li[data-tipo='post_compra']")
        for j in range(await li.count()):
            fila = li.nth(j)
            resenas.append({
                "id": await fila.get_attribute("data-comentario-id"),
                "orden_id": orden_id,
                "cliente_id": await fila.get_attribute("data-cliente-id"),
                "producto_id": await fila.get_attribute("data-producto-id"),
                "calificacion": await fila.get_attribute("data-calificacion"),
                "fecha": await fila.get_attribute("data-fecha"),
                "texto": (await fila.locator("[itemprop='reviewBody']").inner_text()).strip(),
            })

    print(f"  página {numero_pagina:>2}: {n} órdenes")
    siguiente = pagina.locator('nav[aria-label="Paginacion"] a:has-text("Siguiente")')
    if await siguiente.count() == 0:
        break
    numero_pagina += 1

print(f"\n{len(resenas)} reseñas de entrega en {numero_pagina} páginas de órdenes")

  página  1: 10 órdenes
  página  2: 10 órdenes
  página  3: 10 órdenes
  página  4: 10 órdenes
  página  5: 10 órdenes
  página  6: 10 órdenes
  página  7: 10 órdenes
  página  8: 10 órdenes
  página  9: 10 órdenes
  página 10: 10 órdenes
  página 11: 10 órdenes
  página 12: 10 órdenes
  página 13: 10 órdenes
  página 14: 10 órdenes
  página 15: 10 órdenes
  página 16: 10 órdenes
  página 17: 10 órdenes
  página 18: 10 órdenes
  página 19: 10 órdenes
  página 20: 10 órdenes
  página 21: 10 órdenes
  página 22: 10 órdenes
  página 23: 10 órdenes
  página 24: 10 órdenes
  página 25: 10 órdenes
  página 26: 10 órdenes
  página 27: 10 órdenes
  página 28: 10 órdenes
  página 29: 10 órdenes
  página 30: 10 órdenes

110 reseñas de entrega en 30 páginas de órdenes


## Reutilizar la sesión guardada

Un contexto nuevo creado con `storage_state=STATE_PATH` entra directo a una página
protegida (`/admin/api-keys`) **sin volver a pasar por el login**.

In [5]:
contexto2 = await navegador.new_context(storage_state=STATE_PATH, user_agent=UA)
pagina2 = await contexto2.new_page()
await pagina2.goto(f"{BASE}/admin/api-keys", wait_until="networkidle")

print("URL:", pagina2.url, "(no redirige a /login)")
print("Título de la página protegida:", await pagina2.locator("h2").first.inner_text())
print("Filas en la tabla de API keys:", await pagina2.locator("table tbody tr").count())

await contexto2.close()

URL: http://localhost:3000/admin/api-keys (no redirige a /login)
Título de la página protegida: API keys
Filas en la tabla de API keys: 1


## Guardar en `data/playwright_resenas_entrega.csv` y cerrar el navegador

In [6]:
OUTPUT = os.path.join(DATA_DIR, "playwright_resenas_entrega.csv")
with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(resenas[0].keys()))
    writer.writeheader()
    writer.writerows(resenas)

await navegador.close()
await playwright.stop()

print(f"Guardadas {len(resenas)} filas en {OUTPUT}")

Guardadas 110 filas en /Users/erichuiza/Documents/pucp/miería web/2026-2/dev/sesion-de-clase-03/notebooks/../data/playwright_resenas_entrega.csv


Resumen de la sesión: **a)** REST público (recurso, colección, paginación) ->
**b)** REST con seguridad (API key, luego OAuth2) -> **c)** GraphQL (query simple,
luego variables + anidado + OAuth) -> **d)** sin API, navegador real con login.